<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/01_Basic/L11_Prompt_Engineering_101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L11: Prompt Engineering 101 - Crafting Effective Instructions

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Basic  
**Lesson:** 11 of 15

---

## Learning Objectives

By the end of this lesson, you will:
- Understand the fundamentals of prompt engineering
- Master prompt design patterns and best practices
- Learn instruction formatting techniques
- Create effective few-shot examples
- Build reusable prompt templates
- Apply prompting strategies for different tasks
- Optimize prompts for better model outputs

---

## Concept: What is Prompt Engineering?

**Prompt engineering** is the art and science of designing inputs to get desired outputs from language models.

### Why Prompt Engineering Matters:

**The Challenge:**
- Language models are powerful but need clear instructions
- Same question, different phrasing = different results
- Models can't read your mind

**The Solution:**
- Well-crafted prompts guide model behavior
- Structured instructions improve consistency
- Examples teach models what you want

### The Prompt Anatomy:

```
[System Context] (Optional)
[Instruction] (What to do)
[Input Data] (What to process)
[Output Format] (How to respond)
[Examples] (Optional demonstrations)
```

### Key Principles:

1. **Be Specific** - Clear instructions beat vague ones
2. **Provide Context** - Give the model relevant information
3. **Show Examples** - Demonstrate desired behavior
4. **Format Matters** - Structure helps the model understand
5. **Iterate** - Test and refine your prompts

---

In [ ]:
# Step 1: Install and Import Required Libraries
!pip install transformers torch accelerate -q

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# Step 2: Load Model and Tokenizer

print("Loading GPT-2 Model\n")
print("=" * 60)

# Load GPT-2 for text generation
model_name = "gpt2"
print(f"Model: {model_name}\n")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set padding token
tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded successfully!")
print(f"Model type: {type(model).__name__}")
print(f"Number of parameters: {model.num_parameters():,}")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")
print()

print("We'll use this model to explore different prompting techniques.")

In [ ]:
# Helper function for text generation
def generate_text(prompt, max_length=100, temperature=0.7, top_p=0.9):
    """Generate text from a prompt with specified parameters."""
    inputs = tokenizer(prompt, return_tensors="pt")
    
    with torch.no_grad():
        outputs = model.generate(
            inputs['input_ids'],
            max_length=max_length,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2
        )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Helper function defined!")
print("\nUsage: generate_text(prompt, max_length, temperature, top_p)")

## Part 1: Basic Prompt Design Patterns

**Prompt design patterns** are proven structures that consistently produce good results.

### Pattern 1: Direct Instruction

Simply tell the model what to do.

```
Instruction: [Clear command]
Input: [Data to process]
```

### Pattern 2: Role-Based Prompting

Assign the model a specific role or persona.

```
You are a [role]. [Task description]
```

### Pattern 3: Step-by-Step Reasoning

Ask the model to think through the problem.

```
Let's solve this step by step:
1. [First step]
2. [Second step]
...
```

### Pattern 4: Format Specification

Define exactly how the output should look.

```
Provide your answer in the following format:
- Field 1: [value]
- Field 2: [value]
```

---

In [ ]:
# Step 3: Comparing Prompt Design Patterns

print("Prompt Design Patterns Comparison\n")
print("=" * 60)

# Pattern 1: Vague prompt (baseline)
print("\n1. VAGUE PROMPT (Baseline)")
print("-" * 60)
vague_prompt = "Write about artificial intelligence"
print(f"Prompt: {vague_prompt}\n")
result = generate_text(vague_prompt, max_length=80)
print(f"Output: {result}")
print()

# Pattern 2: Direct instruction
print("\n2. DIRECT INSTRUCTION")
print("-" * 60)
direct_prompt = """Write a concise definition of artificial intelligence in one sentence."""
print(f"Prompt: {direct_prompt}\n")
result = generate_text(direct_prompt, max_length=80)
print(f"Output: {result}")
print()

# Pattern 3: Role-based prompting
print("\n3. ROLE-BASED PROMPTING")
print("-" * 60)
role_prompt = """You are a computer science professor. Explain artificial intelligence to a beginner student."""
print(f"Prompt: {role_prompt}\n")
result = generate_text(role_prompt, max_length=100)
print(f"Output: {result}")
print()

# Pattern 4: Format specification
print("\n4. FORMAT SPECIFICATION")
print("-" * 60)
format_prompt = """Define artificial intelligence using this format:
Definition: [one sentence]
Key components: [list 3 items]
Example application: [one example]"""
print(f"Prompt: {format_prompt}\n")
result = generate_text(format_prompt, max_length=120)
print(f"Output: {result}")
print()

print("=" * 60)
print("\nObservation: More specific prompts produce more targeted outputs!")

## Part 2: Instruction Formatting Techniques

**Instruction formatting** helps models understand exactly what you want.

### Key Formatting Principles:

1. **Use Clear Delimiters**
   - Separate instruction from input
   - Use markers like ###, ---, or labels

2. **Structure with Labels**
   - Instruction:
   - Input:
   - Output:

3. **Numbered Steps**
   - Break complex tasks into steps
   - Guide the model's reasoning

4. **Explicit Constraints**
   - Specify length, tone, format
   - Define what to include/exclude

### Good vs Bad Formatting:

**Bad:**
```
Translate this to French hello how are you
```

**Good:**
```
Instruction: Translate the following English text to French.
Input: Hello, how are you?
Output:
```

---

In [ ]:
# Step 4: Instruction Formatting Examples

print("Instruction Formatting Techniques\n")
print("=" * 60)

# Example 1: Poor formatting
print("\n1. POOR FORMATTING")
print("-" * 60)
poor_prompt = "summarize this text Machine learning is a subset of AI"
print(f"Prompt: {poor_prompt}\n")
result = generate_text(poor_prompt, max_length=60)
print(f"Output: {result}")
print()

# Example 2: Clear delimiters
print("\n2. CLEAR DELIMITERS")
print("-" * 60)
delimited_prompt = """Task: Summarize the following text in one sentence.

###
Machine learning is a subset of artificial intelligence that enables computers to learn from data without being explicitly programmed. It uses algorithms to identify patterns and make predictions.
###

Summary:"""
print(f"Prompt: {delimited_prompt}\n")
result = generate_text(delimited_prompt, max_length=100)
print(f"Output: {result}")
print()

# Example 3: Structured labels
print("\n3. STRUCTURED LABELS")
print("-" * 60)
labeled_prompt = """Instruction: Extract the key information from the text.
Input: Python was created by Guido van Rossum in 1991.
Output format: Creator, Year, Language
Output:"""
print(f"Prompt: {labeled_prompt}\n")
result = generate_text(labeled_prompt, max_length=80)
print(f"Output: {result}")
print()

# Example 4: Explicit constraints
print("\n4. EXPLICIT CONSTRAINTS")
print("-" * 60)
constrained_prompt = """Write a product description with these requirements:
- Length: Exactly 2 sentences
- Tone: Professional
- Product: Wireless headphones
- Include: Battery life and comfort

Description:"""
print(f"Prompt: {constrained_prompt}\n")
result = generate_text(constrained_prompt, max_length=100)
print(f"Output: {result}")
print()

print("=" * 60)
print("\nClear formatting leads to better, more predictable outputs!")

## Part 3: Zero-Shot Prompting

**Zero-shot prompting** means asking the model to perform a task without providing any examples.

### When to Use Zero-Shot:

- Simple, well-defined tasks
- Model already knows the task type
- Quick prototyping
- Limited prompt space

### Zero-Shot Structure:

```
[Clear instruction] + [Input] = [Output]
```

### Best Practices:

1. Be explicit about the task
2. Specify output format
3. Include relevant context
4. Use clear language

---

In [ ]:
# Step 5: Zero-Shot Prompting Examples

print("Zero-Shot Prompting\n")
print("=" * 60)

# Task 1: Sentiment classification
print("\n1. SENTIMENT CLASSIFICATION")
print("-" * 60)
sentiment_prompt = """Classify the sentiment of the following review as Positive, Negative, or Neutral.

Review: The product exceeded my expectations. Great quality and fast shipping!

Sentiment:"""
print(f"Prompt: {sentiment_prompt}\n")
result = generate_text(sentiment_prompt, max_length=80, temperature=0.3)
print(f"Output: {result}")
print()

# Task 2: Entity extraction
print("\n2. ENTITY EXTRACTION")
print("-" * 60)
entity_prompt = """Extract the person name, company, and location from the text.

Text: Sarah Johnson works at Microsoft in Seattle.

Person:
Company:
Location:"""
print(f"Prompt: {entity_prompt}\n")
result = generate_text(entity_prompt, max_length=80, temperature=0.3)
print(f"Output: {result}")
print()

# Task 3: Text transformation
print("\n3. TEXT TRANSFORMATION")
print("-" * 60)
transform_prompt = """Rewrite the following sentence in a more formal tone.

Original: Hey, can you send me that report ASAP?

Formal version:"""
print(f"Prompt: {transform_prompt}\n")
result = generate_text(transform_prompt, max_length=80, temperature=0.5)
print(f"Output: {result}")
print()

# Task 4: Question answering
print("\n4. QUESTION ANSWERING")
print("-" * 60)
qa_prompt = """Answer the question based on the context.

Context: The Eiffel Tower was completed in 1889 and stands 330 meters tall.
Question: How tall is the Eiffel Tower?

Answer:"""
print(f"Prompt: {qa_prompt}\n")
result = generate_text(qa_prompt, max_length=80, temperature=0.3)
print(f"Output: {result}")
print()

print("=" * 60)
print("\nZero-shot works well for straightforward tasks!")

## Part 4: Few-Shot Prompting

**Few-shot prompting** provides examples to demonstrate the desired behavior.

### When to Use Few-Shot:

- Complex or ambiguous tasks
- Specific output format needed
- Consistent style required
- Zero-shot performance is poor

### Few-Shot Structure:

```
[Instruction]

Example 1:
Input: [example input 1]
Output: [example output 1]

Example 2:
Input: [example input 2]
Output: [example output 2]

Now your turn:
Input: [actual input]
Output:
```

### Example Selection Tips:

1. **Diverse examples** - Cover different cases
2. **Clear patterns** - Make the task obvious
3. **Consistent format** - Same structure for all
4. **Quality over quantity** - 2-5 good examples usually enough

---

In [ ]:
# Step 6: Few-Shot Prompting Examples

print("Few-Shot Prompting\n")
print("=" * 60)

# Task 1: Custom classification
print("\n1. CUSTOM CLASSIFICATION (Product Categories)")
print("-" * 60)
few_shot_classify = """Classify products into categories: Electronics, Clothing, or Food.

Example 1:
Product: Laptop computer
Category: Electronics

Example 2:
Product: Cotton t-shirt
Category: Clothing

Example 3:
Product: Organic apples
Category: Food

Now classify:
Product: Wireless mouse
Category:"""
print(f"Prompt: {few_shot_classify}\n")
result = generate_text(few_shot_classify, max_length=120, temperature=0.3)
print(f"Output: {result}")
print()

# Task 2: Format conversion
print("\n2. FORMAT CONVERSION")
print("-" * 60)
few_shot_format = """Convert casual messages to professional emails.

Example 1:
Casual: Hey, meeting at 3?
Professional: Dear colleague, shall we schedule our meeting for 3 PM?

Example 2:
Casual: Got the files, thx!
Professional: Thank you for sending the files. I have received them.

Now convert:
Casual: Can u review my doc?
Professional:"""
print(f"Prompt: {few_shot_format}\n")
result = generate_text(few_shot_format, max_length=100, temperature=0.5)
print(f"Output: {result}")
print()

# Task 3: Pattern completion
print("\n3. PATTERN COMPLETION")
print("-" * 60)
few_shot_pattern = """Generate creative product names following the pattern.

Example 1:
Product: Coffee maker
Name: BrewMaster Pro

Example 2:
Product: Running shoes
Name: SprintForce Elite

Example 3:
Product: Desk lamp
Name: LumiDesk Plus

Now create:
Product: Water bottle
Name:"""
print(f"Prompt: {few_shot_pattern}\n")
result = generate_text(few_shot_pattern, max_length=100, temperature=0.7)
print(f"Output: {result}")
print()

print("=" * 60)
print("\nFew-shot examples teach the model your specific requirements!")

## Part 5: Chain-of-Thought Prompting

**Chain-of-Thought (CoT)** prompting encourages the model to show its reasoning process.

### Why Chain-of-Thought Works:

- Breaks complex problems into steps
- Reduces reasoning errors
- Makes outputs more interpretable
- Improves accuracy on multi-step tasks

### CoT Patterns:

**Pattern 1: Explicit Steps**
```
Let's solve this step by step:
1. [First step]
2. [Second step]
3. [Conclusion]
```

**Pattern 2: Think Aloud**
```
Let's think through this carefully:
[Reasoning process]
Therefore, [answer]
```

**Pattern 3: Few-Shot CoT**
```
Example with reasoning:
Question: [Q]
Thinking: [reasoning]
Answer: [A]
```

---

In [ ]:
# Step 7: Chain-of-Thought Prompting

print("Chain-of-Thought Prompting\n")
print("=" * 60)

# Without CoT (baseline)
print("\n1. WITHOUT CHAIN-OF-THOUGHT (Baseline)")
print("-" * 60)
no_cot = """Question: If a store has 15 apples and sells 7, then receives 12 more, how many apples does it have?

Answer:"""
print(f"Prompt: {no_cot}\n")
result = generate_text(no_cot, max_length=80, temperature=0.3)
print(f"Output: {result}")
print()

# With explicit CoT
print("\n2. WITH CHAIN-OF-THOUGHT")
print("-" * 60)
with_cot = """Question: If a store has 15 apples and sells 7, then receives 12 more, how many apples does it have?

Let's solve this step by step:
1. Starting apples: 15
2. After selling 7: 15 - 7 = 8
3. After receiving 12 more: 8 + 12 = 20

Answer:"""
print(f"Prompt: {with_cot}\n")
result = generate_text(with_cot, max_length=100, temperature=0.3)
print(f"Output: {result}")
print()

# Few-shot CoT
print("\n3. FEW-SHOT CHAIN-OF-THOUGHT")
print("-" * 60)
few_shot_cot = """Solve word problems by showing your reasoning.

Example:
Question: A book costs 12 dollars. If you buy 3 books, how much do you spend?
Thinking: Each book is 12 dollars. For 3 books, I multiply: 3 × 12 = 36 dollars.
Answer: 36 dollars

Now solve:
Question: A pizza has 8 slices. If 3 people each eat 2 slices, how many slices remain?
Thinking:"""
print(f"Prompt: {few_shot_cot}\n")
result = generate_text(few_shot_cot, max_length=120, temperature=0.5)
print(f"Output: {result}")
print()

print("=" * 60)
print("\nChain-of-thought improves reasoning and accuracy!")

## Part 6: Prompt Templates

**Prompt templates** are reusable structures that you can fill in with different inputs.

### Benefits of Templates:

1. **Consistency** - Same format every time
2. **Efficiency** - Write once, use many times
3. **Maintainability** - Easy to update and improve
4. **Scalability** - Process multiple inputs easily

### Template Structure:

```python
template = """
[Fixed instruction]
{variable_1}
{variable_2}
[Fixed output format]
"""

filled_prompt = template.format(
    variable_1="value1",
    variable_2="value2"
)
```

---

In [ ]:
# Step 8: Creating and Using Prompt Templates

print("Prompt Templates\n")
print("=" * 60)

# Template 1: Summarization template
print("\n1. SUMMARIZATION TEMPLATE")
print("-" * 60)

summarization_template = """Summarize the following text in {num_sentences} sentences.

Text:
{text}

Summary:"""

# Use the template with different inputs
texts = [
    "Artificial intelligence is transforming industries worldwide. From healthcare to finance, AI systems are improving efficiency and decision-making. Machine learning algorithms can now detect diseases, predict market trends, and automate complex tasks.",
    "Climate change poses significant challenges to our planet. Rising temperatures are causing ice caps to melt and sea levels to rise. Scientists emphasize the urgent need for sustainable practices and renewable energy adoption."
]

for i, text in enumerate(texts, 1):
    prompt = summarization_template.format(num_sentences=2, text=text)
    print(f"Example {i}:")
    print(f"Original length: {len(text)} characters")
    result = generate_text(prompt, max_length=150, temperature=0.5)
    print(f"Summary: {result.split('Summary:')[-1].strip() if 'Summary:' in result else result}")
    print()

print("-" * 60)

In [ ]:
# Template 2: Classification template
print("\n2. CLASSIFICATION TEMPLATE")
print("=" * 60)

classification_template = """Classify the following {item_type} into one of these categories: {categories}

{item_type}: {item}

Category:"""

# Use for different classification tasks
classification_tasks = [
    {
        "item_type": "email",
        "categories": "Urgent, Important, Spam",
        "item": "Your account will be closed unless you verify your information immediately!"
    },
    {
        "item_type": "movie genre",
        "categories": "Action, Comedy, Drama, Horror",
        "item": "A group of friends navigate life's challenges while supporting each other through ups and downs."
    },
    {
        "item_type": "customer feedback",
        "categories": "Positive, Negative, Neutral",
        "item": "The product works as expected but shipping took longer than promised."
    }
]

for i, task in enumerate(classification_tasks, 1):
    prompt = classification_template.format(**task)
    print(f"\nTask {i}: Classify {task['item_type']}")
    result = generate_text(prompt, max_length=100, temperature=0.3)
    print(f"Result: {result.split('Category:')[-1].strip()[:50] if 'Category:' in result else result[:50]}")

print("\n" + "=" * 60)

In [ ]:
# Template 3: Creative generation template
print("\n3. CREATIVE GENERATION TEMPLATE")
print("=" * 60)

creative_template = """Write a {length} {content_type} about {topic} in a {tone} tone.

{content_type.capitalize()}:"""

# Generate different creative content
creative_tasks = [
    {"length": "short", "content_type": "story", "topic": "a robot learning to paint", "tone": "whimsical"},
    {"length": "brief", "content_type": "poem", "topic": "morning coffee", "tone": "peaceful"},
    {"length": "concise", "content_type": "description", "topic": "a futuristic city", "tone": "exciting"}
]

for i, task in enumerate(creative_tasks, 1):
    prompt = creative_template.format(**task)
    print(f"\nGeneration {i}: {task['content_type'].title()} about {task['topic']}")
    result = generate_text(prompt, max_length=120, temperature=0.8)
    print(f"Output: {result.split(':')[-1].strip()[:200] if ':' in result else result[:200]}")

print("\n" + "=" * 60)
print("\nTemplates make prompt engineering scalable and consistent!")

## Part 7: Task-Specific Prompting Strategies

Different tasks require different prompting approaches.

---

In [ ]:
# Step 9: Task-Specific Prompting

print("Task-Specific Prompting Strategies\n")
print("=" * 60)

# Task 1: Data extraction
print("\n1. DATA EXTRACTION")
print("-" * 60)
extraction_prompt = """Extract structured information from the text.

Text: John Smith, age 35, works as a Software Engineer at TechCorp in San Francisco. His email is john.smith@techcorp.com.

Extract:
- Name:
- Age:
- Job Title:
- Company:
- Location:
- Email:"""
print(f"Prompt: {extraction_prompt}\n")
result = generate_text(extraction_prompt, max_length=150, temperature=0.2)
print(f"Output: {result}")
print()

# Task 2: Text rewriting
print("\n2. TEXT REWRITING")
print("-" * 60)
rewriting_prompt = """Rewrite the following text to be more concise while keeping the key information.

Original: Due to the fact that we have received numerous complaints from customers regarding the quality of the product, we have made the decision to issue a full recall of all units that were manufactured during the month of January.

Concise version:"""
print(f"Prompt: {rewriting_prompt}\n")
result = generate_text(rewriting_prompt, max_length=120, temperature=0.5)
print(f"Output: {result}")
print()

# Task 3: Comparison
print("\n3. COMPARISON TASK")
print("-" * 60)
comparison_prompt = """Compare the following two options and provide a brief analysis.

Option A: Electric car - Zero emissions, higher upfront cost, lower maintenance
Option B: Hybrid car - Lower emissions, moderate cost, standard maintenance

Comparison:"""
print(f"Prompt: {comparison_prompt}\n")
result = generate_text(comparison_prompt, max_length=150, temperature=0.6)
print(f"Output: {result}")
print()

# Task 4: Code explanation
print("\n4. CODE EXPLANATION")
print("-" * 60)
code_prompt = """Explain what this Python code does in simple terms.

Code:
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

Explanation:"""
print(f"Prompt: {code_prompt}\n")
result = generate_text(code_prompt, max_length=120, temperature=0.4)
print(f"Output: {result}")
print()

print("=" * 60)
print("\nTailor your prompts to the specific task for best results!")

## Part 8: Prompt Optimization Techniques

**Prompt optimization** is the iterative process of improving prompts for better results.

### Optimization Strategies:

1. **A/B Testing**
   - Test multiple prompt variations
   - Compare outputs systematically
   - Choose the best performer

2. **Parameter Tuning**
   - Adjust temperature for creativity vs consistency
   - Modify max_length for output size
   - Tune top_p for diversity

3. **Iterative Refinement**
   - Start simple, add complexity as needed
   - Add constraints based on failures
   - Incorporate feedback

4. **Context Management**
   - Include only relevant information
   - Remove unnecessary details
   - Balance context vs instruction

---

In [ ]:
# Step 10: Prompt Optimization Example

print("Prompt Optimization Process\n")
print("=" * 60)

task = "Generate a professional email declining a meeting request"

# Version 1: Basic prompt
print("\nVERSION 1: Basic Prompt")
print("-" * 60)
v1_prompt = "Write an email declining a meeting."
print(f"Prompt: {v1_prompt}\n")
result = generate_text(v1_prompt, max_length=100, temperature=0.7)
print(f"Output: {result}")
print()

# Version 2: Add context
print("\nVERSION 2: Added Context")
print("-" * 60)
v2_prompt = """Write a professional email declining a meeting request due to schedule conflict."""
print(f"Prompt: {v2_prompt}\n")
result = generate_text(v2_prompt, max_length=100, temperature=0.7)
print(f"Output: {result}")
print()

# Version 3: Add structure
print("\nVERSION 3: Added Structure")
print("-" * 60)
v3_prompt = """Write a professional email with the following structure:
- Polite greeting
- Acknowledge the meeting request
- Decline due to schedule conflict
- Suggest alternative
- Professional closing

Email:"""
print(f"Prompt: {v3_prompt}\n")
result = generate_text(v3_prompt, max_length=150, temperature=0.6)
print(f"Output: {result}")
print()

# Version 4: Add example (few-shot)
print("\nVERSION 4: Added Example")
print("-" * 60)
v4_prompt = """Write a professional email declining a meeting request.

Example:
Subject: Re: Meeting Request

Dear [Name],

Thank you for the meeting invitation. Unfortunately, I have a scheduling conflict during that time. Would next Tuesday at 2 PM work for you instead?

Best regards,
[Your name]

Now write a similar email for declining a lunch meeting:"""
print(f"Prompt: {v4_prompt}\n")
result = generate_text(v4_prompt, max_length=150, temperature=0.6)
print(f"Output: {result}")
print()

print("=" * 60)
print("\nObservation: Each iteration produces more targeted, useful output!")
print("\nOptimization Tips:")
print("  1. Start simple, add complexity gradually")
print("  2. Test with multiple examples")
print("  3. Adjust temperature based on task (lower for factual, higher for creative)")
print("  4. Use examples when format is important")
print("  5. Be specific about constraints and requirements")

## Part 9: Common Pitfalls and Best Practices

**Avoiding common mistakes** improves prompt effectiveness.

---

In [ ]:
# Step 11: Common Pitfalls and Solutions

import pandas as pd

print("Common Prompt Engineering Pitfalls\n")
print("=" * 60)

# Create comparison table
data = {
    'Pitfall': [
        'Vague instructions',
        'Too much context',
        'Ambiguous phrasing',
        'No output format',
        'Inconsistent examples',
        'Wrong temperature'
    ],
    'Problem': [
        'Model guesses intent',
        'Important info gets lost',
        'Multiple interpretations',
        'Unpredictable structure',
        'Model learns wrong pattern',
        'Output too random or boring'
    ],
    'Solution': [
        'Be specific and explicit',
        'Include only relevant details',
        'Use clear, direct language',
        'Specify exact format needed',
        'Use consistent, clear examples',
        'Tune based on task type'
    ]
}

df = pd.DataFrame(data)
print(df.to_string(index=False))
print("\n" + "=" * 60)

print("\n\nBest Practices Summary:\n")
print("-" * 60)

best_practices = [
    "1. Start with clear, specific instructions",
    "2. Use consistent formatting and structure",
    "3. Provide 2-5 high-quality examples for complex tasks",
    "4. Specify output format explicitly",
    "5. Test with multiple inputs to verify consistency",
    "6. Iterate and refine based on results",
    "7. Use lower temperature (0.2-0.5) for factual tasks",
    "8. Use higher temperature (0.7-1.0) for creative tasks",
    "9. Include constraints and requirements upfront",
    "10. Keep prompts as simple as possible while being effective"
]

for practice in best_practices:
    print(f"  {practice}")

print("\n" + "=" * 60)

In [ ]:
# Step 12: Before and After Examples

print("\nBefore and After: Prompt Improvements\n")
print("=" * 60)

examples = [
    {
        'task': 'Product Description',
        'before': 'Write about headphones',
        'after': 'Write a 2-sentence product description for wireless headphones, highlighting comfort and battery life in a professional tone.'
    },
    {
        'task': 'Data Analysis',
        'before': 'Analyze this data',
        'after': 'Analyze the following sales data and provide: 1) Total revenue, 2) Best-selling product, 3) One key insight. Format as bullet points.'
    },
    {
        'task': 'Translation',
        'before': 'Translate to Spanish',
        'after': 'Translate the following English text to Spanish, maintaining a formal tone:\n\nText: [input]\n\nSpanish translation:'
    },
    {
        'task': 'Code Review',
        'before': 'Review this code',
        'after': 'Review the following Python code and provide: 1) What it does, 2) Potential bugs, 3) Improvement suggestions. Be concise.'
    }
]

for i, ex in enumerate(examples, 1):
    print(f"\nExample {i}: {ex['task']}")
    print("-" * 60)
    print(f"Before: {ex['before']}")
    print(f"After:  {ex['after']}")
    print()

print("=" * 60)
print("\nKey Improvements:")
print("  - Specific task definition")
print("  - Clear output requirements")
print("  - Explicit constraints")
print("  - Structured format")

## Exercises

### Exercise 1: Prompt Design Patterns
Create prompts using different design patterns for the same task:
```python
task = "Classify customer reviews as Positive, Negative, or Neutral"
# Create 4 versions:
# 1. Direct instruction (zero-shot)
# 2. Role-based prompting
# 3. Few-shot with examples
# 4. Chain-of-thought reasoning
# Compare the outputs and effectiveness
```

### Exercise 2: Template Creation
Build a reusable template for email generation:
```python
# Create a template that accepts:
# - email_type (request, response, follow-up)
# - tone (formal, casual, friendly)
# - key_points (list of main points to include)
# - recipient_name
# Test with at least 3 different combinations
```

### Exercise 3: Prompt Optimization
Optimize a prompt through iterations:
```python
starting_prompt = "Summarize this article"
# Create 5 improved versions, each adding:
# - More specific instructions
# - Output format requirements
# - Length constraints
# - Tone specifications
# - Example demonstration
# Document what each iteration improves
```

### Exercise 4: Task-Specific Prompting
Create specialized prompts for different tasks:
- Data extraction from unstructured text
- Code generation with specific requirements
- Creative story writing with constraints
- Technical documentation simplification

Test each prompt with multiple inputs and refine.

### Exercise 5: Few-Shot Learning
Design a few-shot prompt for a custom task:
```python
# Task: Convert technical jargon to plain language
# Create 3-5 high-quality examples
# Test on new technical terms
# Analyze: How many examples are needed for good performance?
```

### Exercise 6: Parameter Exploration
Experiment with generation parameters:
```python
prompt = "Write a creative story opening"
# Test combinations of:
# - temperature: [0.3, 0.7, 1.0, 1.5]
# - top_p: [0.7, 0.9, 0.95]
# - max_length: [50, 100, 150]
# Find optimal settings for creative vs coherent output
```

---